<div dir="rtl" align="right">

# مُقارنةُ نواةِ SVM

**مجموعةُ البياناتِ**: MOABB BNCI2014-001 (تخيّلٌ حركيّ)  
**القنواتُ**: 22 قناةً  
**معدّلُ أخذِ العيناتِ**: 250 Hz  
**المُشاركُ**: 1

---

## نظرةٌ عامّةٌ

نُقارنُ SVM بِنواةٍ خطّيّةٍ ومُتعدّدةِ الحدودِ و RBF على سماتِ قُوّةِ النطاقِ لِتصنيفِ التخيّلِ الحركيّ.

## ماذا يَعمَلُ هذا الدفترُ؟

يَحسبُ سماتِ قُوّةِ النطاقِ، ويُقسّمُ إلى تدريب/اختبار، ويُقيّسُ، ويُدرّبُ SVM بِكلِّ نوعِ نواةٍ.

## المُخرجاتُ المُتوقّعةُ

- مخططٌ شريطيٌّ يُقارنُ دقّةَ النواةِ الخطّيّةِ و poly و rbf
- مصفوفةُ الالتباسِ لِلنواةِ الأفضلِ أداءً
- الدقّةُ مطبوعةٌ لِكلِّ نواةٍ

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ |
| --- | --- |
| fmin | 8 |
| fmax | 32 |
| n_classes | 2 |
| kernels | linear, poly, rbf |
| test_size | 0.2 |

</div>


<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>


In [ ]:
!pip install moabb mne scipy numpy plotly scikit-learn


<div dir="rtl" align="right">

## 2. تحميلُ مجموعةِ بياناتِ MOABB

تُنزّلُ MOABB البياناتِ تلقائيّاً عندَ أوّلِ استدعاءٍ (حوالي 44 ميجابايت).

</div>


In [ ]:
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery
import numpy as np

dataset = BNCI2014_001()
paradigm = MotorImagery(n_classes=2, fmin=8, fmax=32)
X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[1])

print(f'X shape: {X.shape}')
print(f'Labels: {np.unique(labels)}')
print(f'Trials: {len(labels)}')


In [ ]:
mask = (labels == 'left_hand') | (labels == 'right_hand')
X = X[mask]
labels = labels[mask]

print(f'After filtering - X shape: {X.shape}')
print(f'Labels: {np.unique(labels)}')


<div dir="rtl" align="right">

## 3. استكشافُ البياناتِ

</div>


In [ ]:
n_trials, n_channels, n_samples = X.shape
print(f'Trials: {n_trials}')
print(f'Channels: {n_channels}')
print(f'Samples per trial: {n_samples}')
print(f'Trial duration: {n_samples/250:.2f} s')


<div dir="rtl" align="right">

## 4. تدريبُ SVM بِنواةٍ مُختلفةٍ

</div>


In [ ]:
from scipy.signal import welch

FS = 250
BANDS = [(8, 13, 'alpha'), (13, 30, 'beta')]

features = np.zeros((n_trials, n_channels * len(BANDS)))
for trial in range(n_trials):
    for ch in range(n_channels):
        freqs, psd = welch(X[trial, ch, :], fs=FS, nperseg=256)
        for b_idx, (fmin, fmax, bname) in enumerate(BANDS):
            mask_f = (freqs >= fmin) & (freqs <= fmax)
            features[trial, ch * len(BANDS) + b_idx] = np.trapezoid(psd[mask_f], freqs[mask_f])

print(f'Feature matrix shape: {features.shape}')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.2, random_state=42, stratify=labels
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix

KERNELS = ['linear', 'poly', 'rbf']
accuracies = []
best_kernel = KERNELS[0]
best_acc = 0.0
best_cm = None

for kernel in KERNELS:
    clf = SVC(kernel=kernel, random_state=42)
    clf.fit(X_train_scaled, y_train)
    y_pred = clf.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    accuracies.append(acc)
    print(f'kernel={kernel}: accuracy={acc:.4f}')
    if acc > best_acc:
        best_acc = acc
        best_kernel = kernel
        best_cm = confusion_matrix(y_test, y_pred, labels=['left_hand', 'right_hand'])

print(f'Best kernel: {best_kernel}, accuracy: {best_acc:.4f}')
classes = ['left_hand', 'right_hand']


<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**

- نواةُ RBF غالباً تُؤدّي أفضلَ على بياناتِ EEG بسببِ حدودِ الفئاتِ غيرِ الخطّيّةِ
- النواةُ الخطّيّةُ الأسرعُ لكنّها قد تُفرطُ في التعميمِ معَ بياناتٍ مُعقّدةٍ
- مصفوفةُ الالتباسِ تُظهرُ أيُّ نواةٍ تُعمّمُ أفضلَ

</div>


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=2, cols=1, subplot_titles=(
    'Accuracy per Kernel',
    f'Confusion Matrix (best kernel={best_kernel})'))

fig.add_trace(go.Bar(x=KERNELS, y=accuracies, marker_color=['steelblue', 'orange', 'green'], name='Accuracy'), row=1, col=1)
fig.add_trace(go.Heatmap(z=best_cm, x=classes, y=classes, colorscale='Blues',
    text=best_cm, texttemplate='%{text}', textfont={'size': 16}, name='CM', showscale=True), row=2, col=1)

fig.update_xaxes(title_text='Kernel', row=1, col=1)
fig.update_yaxes(title_text='Test Accuracy', row=1, col=1)
fig.update_xaxes(title_text='Predicted', row=2, col=1)
fig.update_yaxes(title_text='True', row=2, col=1)
fig.update_layout(height=800, showlegend=False, title_text='SVM Classification - Kernel Comparison')
fig.show()


<div dir="rtl" align="right">

## خلاصةٌ

- اختيارُ النواةِ يُؤثّرُ بشكلٍ كبيرٍ على أداءِ SVM في بياناتِ EEG
- نواةُ RBF تَستطيعُ التقاطَ العلاقاتِ غيرِ الخطّيّةِ في سماتِ قُوّةِ النطاقِ
- التقييسُ ضروريٌّ قبلَ تدريبِ SVM

</div>
